In [1]:
ingest_date = None
data_format = None   # "json" or "xml"

StatementMeta(, 72ca20e7-f577-4334-b0a9-d17471f5db95, 3, Finished, Available, Finished, False)

In [2]:
%run 00_config

StatementMeta(, 72ca20e7-f577-4334-b0a9-d17471f5db95, 4, Finished, Available, Finished, True)

In [3]:
import glob, re
import xml.etree.ElementTree as ET
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

fmt = (data_format or fmt).lower()
NS = "{http://hl7.org/fhir}"

def audit_meta():
    w = Window.partitionBy("resource", "ingest_date", "page").orderBy(F.col("extraction_timestamp").desc())
    return (spark.read.table("control_audit").where(f"format = '{fmt}'")
            .withColumn("rn", F.row_number().over(w)).where("rn = 1")
            .selectExpr("resource as resource_type", "ingest_date", "page",
                        "api_url_or_params", "extraction_timestamp", "run_id"))

StatementMeta(, 72ca20e7-f577-4334-b0a9-d17471f5db95, 5, Finished, Available, Finished, False)

In [4]:
def load_json(resource):
    df = spark.read.option("multiline", "true").json(raw_glob(resource, "json"))
    if "entry" not in df.columns:
        return None
    rows = (df.withColumn("source_file", F.input_file_name())
              .withColumn("entry", F.explode("entry"))
              .select("source_file", F.col("entry.resource").alias("r")))
    return (rows
        .withColumn("resource_type", F.lit(resource))
        .withColumn("id", F.col("r.id"))
        .withColumn("version_id", F.col("r.meta.versionId"))
        .withColumn("last_updated", F.col("r.meta.lastUpdated"))
        .withColumn("resource_payload", F.to_json("r"))
        .withColumn("page", F.regexp_extract("source_file", r"_page(\d+)\.json", 1).cast("int"))
        .withColumn("ingest_date", F.regexp_extract("source_file", r"ingest_date=([0-9\-]+)", 1))
        .drop("r"))

StatementMeta(, 72ca20e7-f577-4334-b0a9-d17471f5db95, 6, Finished, Available, Finished, False)

In [5]:
def json_to_xml_path(jp):
    return re.sub(r"\[\d+\]", "", jp.replace("$.", "")).replace(".", "/")

def load_xml(resource):
    spec = FLATTEN[resource]
    flat_names = list(spec.keys())
    rows = []
    for fp in glob.glob(f"{files_mount}/raw_xml/{resource}/ingest_date=*/*.xml"):
        ingest = re.search(r"ingest_date=([0-9\-]+)", fp).group(1)
        page = int(re.search(r"_page(\d+)\.xml", fp).group(1))
        root = ET.fromstring(open(fp, encoding="utf-8").read())
        for entry in root.findall(f"{NS}entry"):
            res = entry.find(f"{NS}resource")
            if res is None:
                continue
            el = list(res)[0]
            def val(path, el=el):
                cur = el
                for p in path.split("/"):
                    cur = cur.find(f"{NS}{p}")
                    if cur is None:
                        return None
                return cur.get("value")
            base = [resource, val("id"), val("meta/versionId"), val("meta/lastUpdated"),
                    ET.tostring(el, encoding="unicode"), page, ingest]
            flat = []
            for name in flat_names:
                v = val(json_to_xml_path(spec[name]))
                if name in REFERENCE_COLS and v:
                    v = re.sub(r"^[A-Za-z]+/", "", v)   # "Patient/123" -> "123"
                flat.append(v)
            rows.append(tuple(base + flat))
    if not rows:
        return None
    schema = StructType(
        [StructField("resource_type", StringType()), StructField("id", StringType()),
         StructField("version_id", StringType()), StructField("last_updated", StringType()),
         StructField("resource_payload", StringType()), StructField("page", IntegerType()),
         StructField("ingest_date", StringType())]
        + [StructField(n, StringType()) for n in flat_names])
    return spark.createDataFrame(rows, schema)

StatementMeta(, 72ca20e7-f577-4334-b0a9-d17471f5db95, 7, Finished, Available, Finished, False)

In [6]:
meta = audit_meta()
for resource in resources:
    df = load_json(resource) if fmt == "json" else load_xml(resource)
    if df is None:
        print(f"{resource}: no {fmt} data")
        continue
    df = (df.join(meta, ["resource_type", "ingest_date", "page"], "left")
            .withColumn("bronze_load_timestamp", F.current_timestamp()))
    table = f"bronze_json_{resource.lower()}" if fmt == "json" else f"bronze_xml_{resource.lower()}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(f"{table}: {df.count()} rows")

StatementMeta(, 72ca20e7-f577-4334-b0a9-d17471f5db95, 8, Finished, Available, Finished, False)

bronze_xml_patient: 110 rows
bronze_xml_encounter: 110 rows
bronze_xml_observation: 110 rows
bronze_xml_condition: 110 rows
